# PULSE Temporal Awareness — Gemma 3 4B Training

**Fine-tune Google Gemma 3 4B with PULSE temporal awareness using Unsloth + QLoRA on free Colab T4 GPU.**

PULSE encodes *how time feels* — not just what the clock says. This notebook teaches Gemma 3 to understand circadian rhythms, cognitive capacity, urgency, sleep debt, and experiential time.

### Why Gemma 3 4B + Unsloth?
- **Gemma 3 4B** is text-only (no wasted VRAM on vision components)
- **Unsloth** reduces memory by ~60% and speeds up training ~2x
- Fits comfortably on free Colab T4 (16GB VRAM)
- Better quality than Qwen 1.5B due to larger parameter count

| | Qwen 1.5B | Gemma 3 4B (this) |
|---|---|---|
| Params | 1.5B | 4B |
| VRAM | ~8GB | ~12GB |
| Training time | ~15min | ~25min |
| Quality | Good baseline | Better reasoning |

**Source:** [github.com/lalomorales22/pulse-temporal](https://github.com/lalomorales22/pulse-temporal)

In [ ]:
# Step 0: Install Unsloth + dependencies
import subprocess, sys

try:
    import unsloth
    print(f"Unsloth already installed: {unsloth.__version__}")
except ImportError:
    print("Installing unsloth...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth"])
    print("Installed! If runtime restarts, just continue from the next cell.")

# Verify
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    try: print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    except: print("VRAM: (unable to query)")

In [ ]:
# Step 1: Login to HuggingFace (to upload the trained model)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Step 2: PULSE temporal data generator (self-contained)
import json, random, numpy as np
from datetime import datetime, timedelta

_COG = np.array([0.20,0.15,0.12,0.10,0.12,0.18,0.30,0.50,0.70,0.85,0.95,0.92,0.85,0.72,0.68,0.75,0.88,0.90,0.82,0.70,0.55,0.40,0.30,0.25], dtype=np.float32)
_NRG = np.array([0.15,0.10,0.08,0.07,0.10,0.20,0.40,0.60,0.75,0.85,0.90,0.88,0.82,0.70,0.65,0.72,0.85,0.88,0.80,0.65,0.50,0.35,0.25,0.18], dtype=np.float32)

def _interp(c, h):
    h0 = int(h) % 24
    return float(c[h0]*(1-(h-int(h))) + c[(h0+1)%24]*(h-int(h)))

def get_phase(h):
    if 6<=h<10: return "morning_ramp"
    if 10<=h<12: return "morning_peak"
    if 12<=h<14: return "post_lunch_dip"
    if 14<=h<17: return "afternoon_peak"
    if 17<=h<20: return "evening_wind_down"
    if 20<=h<23: return "night_transition"
    return "deep_night"


# =====================================================
# DATA GENERATOR
# =====================================================

SYSTEM_TEMPLATE = """You are an AI assistant with temporal awareness through the PULSE temporal embedding system. Before each interaction, you receive a temporal context package describing the current moment — not just the time, but what that time means: circadian phase, cognitive capacity, energy level, urgency, and behavioral context.

Use this temporal awareness naturally in your responses. Don't announce it mechanically — weave it into your reasoning the way a thoughtful colleague would who knows what time it is and what's going on.

Current temporal context:
{temporal_context}"""

CONTEXTS = [
    ("monday_crunch", 2.0, 7, 5), ("critical_deadline", 0.5, 9, 4),
    ("normal_tuesday", None, 3, 7.5), ("focus_day", None, 1, 8),
    ("deadline_tomorrow", 24.0, 5, 7), ("friday_winding", None, 2, 7),
    ("saturday_morning", None, 0, 9), ("sunday_evening", 12.0, 0, 7),
    ("late_night", None, 0, 0), ("early_fresh", None, 0, 8),
    ("post_lunch", None, 4, 7), ("peak_morning", None, 2, 8),
    ("overdue", -2.0, 8, 4), ("vacation", None, 0, 9),
    ("all_nighter", None, 0, 2), ("back_to_back", None, 10, 6),
    ("weekend_project", 48.0, 0, 8), ("interview_prep", 3.0, 2, 6),
    ("launch_day", 1.0, 12, 4), ("recovery_day", None, 0, 10),
]

QUESTIONS = [
    ("Should I start a complex refactoring task right now?", "task_suitability"),
    ("Is this a good time for creative brainstorming?", "task_suitability"),
    ("Should I take a break right now?", "break_advice"),
    ("How much focus can I expect from myself right now?", "cognitive_state"),
    ("What does my current temporal state look like?", "full_state"),
    ("Am I in a good phase for deep work?", "work_phase"),
    ("How urgent is my situation right now?", "urgency_assessment"),
    ("Would I be more productive waiting until tomorrow morning?", "timing_optimization"),
    ("How should I prioritize my remaining tasks today?", "prioritization"),
    ("What kind of tasks should I tackle right now given my state?", "task_matching"),
    ("Should I push through or call it a day?", "endurance_check"),
    ("How does this moment compare to a typical morning?", "relative_state"),
    ("Should I schedule a difficult conversation for this time?", "task_suitability"),
    ("Is my energy level normal for this time of day?", "circadian_comparison"),
    ("Would this be a good time to learn something new?", "task_suitability"),
    ("Can I handle a code review right now?", "task_suitability"),
    ("Should I do the easy tasks first or tackle the hard one?", "prioritization"),
    ("How will I feel in 2 hours compared to now?", "timing_optimization"),
    ("Is this deadline realistic given my current state?", "urgency_assessment"),
    ("What's the best use of the next hour?", "task_matching"),
]


def generate_response(question, q_sub, dt, deadline_str, events, sleep):
    hour = dt.hour
    cog = _interp(_COG, hour + dt.minute/60)
    eng = _interp(_NRG, hour + dt.minute/60)
    has_dl = deadline_str is not None
    hours_left = (datetime.fromisoformat(deadline_str) - dt).total_seconds()/3600 if has_dl else None
    is_overdue = has_dl and hours_left < 0
    is_urgent = has_dl and hours_left is not None and 0 < hours_left < 3
    is_night = hour < 6 or hour >= 22
    is_peak = 10 <= hour < 12 or 14 <= hour < 17
    is_dip = 12 <= hour < 14
    is_low_sleep = sleep < 6

    parts = []
    if q_sub == "task_suitability":
        if any(w in question.lower() for w in ("complex", "refactoring", "deep work", "learn", "difficult", "code review")):
            if is_peak and not is_low_sleep and cog > 0.8:
                parts.append(f"This is an ideal window. Cognitive capacity at {cog:.0%} during {'morning' if hour<12 else 'afternoon'} peak.")
                if is_urgent: parts.append(f"But with the deadline in {hours_left:.1f} hours, prioritize what's due first.")
                elif not has_dl: parts.append("No deadlines pressing. Good time to dive deep.")
            elif is_dip:
                parts.append(f"Post-lunch dip — cognitive capacity around {cog:.0%}. Complex tasks have more errors now. Wait until ~3pm when afternoon focus returns.")
            elif is_night:
                parts.append(f"It's {dt.strftime('%I:%M %p')}, cognition at {cog:.0%}. Complex work at this hour creates more bugs than it fixes.")
                if is_low_sleep: parts.append(f"Only {sleep:.0f}h sleep makes this worse. Save it for tomorrow morning.")
            elif is_low_sleep:
                parts.append(f"With {sleep:.0f}h sleep, cognitive capacity is compromised. Stick to routine tasks today.")
            else:
                parts.append(f"Moderate capacity at {cog:.0%}. You could start, but this isn't your peak window.")
        elif "creative" in question.lower() or "brainstorm" in question.lower():
            if is_dip or is_night:
                parts.append("Reduced executive function can actually help creativity — your inner critic is quieter. Good time for brainstorming.")
            elif is_peak:
                parts.append(f"Peak capacity ({cog:.0%}) is great for structured creative work. For wild brainstorming, the afternoon dip might actually work better.")
        elif "break" in question.lower():
            if cog < 0.5 or eng < 0.4:
                parts.append(f"Yes. Energy at {eng:.0%}, cognition at {cog:.0%}. A 15-20 minute break would help.")
            elif is_dip:
                parts.append("Natural post-lunch dip. A short walk now aligns with your body's rhythm.")
            elif events > 5:
                parts.append(f"{events} events today means serious context-switching fatigue. Break would help.")
            else:
                parts.append(f"Energy ({eng:.0%}) and cognition ({cog:.0%}) are solid. Keep going if you're in flow.")
        elif "difficult conversation" in question.lower():
            if is_peak and not is_low_sleep:
                parts.append(f"Cognitive capacity at {cog:.0%} helps with emotional regulation. Reasonable window for it.")
            else:
                parts.append(f"With cognition at {cog:.0%}, you're more likely to be reactive than reflective. Postpone if possible.")

    elif q_sub in ("full_state", "cognitive_state", "work_phase"):
        parts.append(f"It's {dt.strftime('%A %I:%M %p')}.")
        if is_peak: parts.append(f"Peak cognitive window — {cog:.0%} capacity, {eng:.0%} energy. Prime time for demanding work.")
        elif is_dip: parts.append(f"Post-lunch dip. Cognition {cog:.0%}, energy {eng:.0%}. Passes around 2:30-3pm.")
        elif is_night: parts.append(f"Deep night. Cognition {cog:.0%}, energy {eng:.0%}. Your body wants rest.")
        else: parts.append(f"Cognitive capacity {cog:.0%}, energy {eng:.0%}.")
        if is_low_sleep: parts.append(f"Sleep deficit ({sleep:.0f}h) dragging everything down. Expect ~20% more errors.")
        if is_overdue: parts.append(f"Deadline passed {-hours_left:.1f}h ago. High stress.")
        elif is_urgent: parts.append(f"Deadline in {hours_left:.1f}h. Focused execution mode.")

    elif q_sub == "urgency_assessment":
        if is_overdue: parts.append(f"Critical. Deadline passed {-hours_left:.1f}h ago. Damage control mode.")
        elif is_urgent: parts.append(f"High — {hours_left:.1f}h until deadline. This should be your only focus.")
        elif has_dl and hours_left < 24: parts.append(f"Moderate. Deadline {hours_left:.1f}h away. Start planning.")
        elif has_dl: parts.append(f"Low for now — {hours_left:.1f}h out. Keep it on radar.")
        else: parts.append("No active deadlines. Choose work based on energy and interest.")

    elif q_sub == "timing_optimization":
        if cog < 0.5: parts.append("Yes. Tomorrow 10-12am would give roughly double your current capacity.")
        elif is_peak: parts.append("You're in a good window now. Waiting means losing momentum and context.")
        elif is_urgent: parts.append(f"Deadline in {hours_left:.1f}h. Waiting isn't an option.")
        else: parts.append(f"Current {cog:.0%} vs tomorrow's ~93% peak. Depends on task complexity.")

    elif q_sub == "prioritization":
        if is_urgent: parts.append(f"Deadline work first — {hours_left:.1f}h left. Everything else secondary.")
        elif is_peak: parts.append("Use this peak for your hardest task. Save routine work for the dip.")
        elif is_dip: parts.append("Good for: emails, code review, admin. Save complex work for the 3-4pm peak.")
        else: parts.append(f"At {cog:.0%} capacity, match tasks to state. Routine now, demanding later.")

    elif q_sub == "task_matching":
        if cog > 0.8: parts.append("Strong state. Go for: complex debugging, architecture decisions, learning new concepts.")
        elif cog > 0.5: parts.append("Moderate. Good for: code review, incremental features, documentation, discussions.")
        else: parts.append("Low state. Stick to: email triage, filing issues, light reading, planning tomorrow.")

    elif q_sub == "endurance_check":
        if eng < 0.3: parts.append(f"Call it. Energy {eng:.0%}, cognition {cog:.0%}. Past diminishing returns.")
        elif is_urgent: parts.append(f"Push through — {hours_left:.1f}h to deadline. Take a 5-min reset first.")
        elif is_dip: parts.append("Feels like a wall but it's the circadian dip. 15-min break usually restores enough.")
        elif events > 7: parts.append(f"{events} events today. Context-switching cost has accumulated. You've earned the stop.")
        else: parts.append(f"Energy {eng:.0%}, cognition {cog:.0%}. Some runway left if work is engaging.")

    elif q_sub in ("relative_state", "circadian_comparison"):
        parts.append(f"At {dt.strftime('%I:%M %p')}, typical capacity is ~{cog:.0%}.")
        if is_low_sleep: parts.append(f"Your {sleep:.0f}h sleep puts you below baseline. Well-rested you'd be closer to {min(cog+0.15,0.95):.0%}.")
        if is_peak: parts.append("This is normally productive. " + ("Good shape to use it." if not is_low_sleep else "Sleep deficit eating into your best hours."))
        elif is_dip: parts.append("Post-lunch dip is universal. Not a you problem, it's biology.")

    if not parts:
        parts.append(f"Current: {cog:.0%} cognitive, {eng:.0%} energy, {dt.strftime('%A %I:%M %p')}.")
    return " ".join(parts)


def generate_dataset(n=2000, seed=42):
    rng = random.Random(seed)
    examples = []
    for _ in range(n):
        name, dl_off, events, sleep = rng.choice(CONTEXTS)
        base = datetime(2026, rng.randint(1,12), rng.randint(1,28))
        hour = rng.randint(0, 23)
        minute = rng.choice([0,15,30,45])
        dt = base.replace(hour=hour, minute=minute)
        if "late_night" in name or "all_nighter" in name: dt = dt.replace(hour=rng.choice([0,1,2,3,23]))
        elif "early" in name: dt = dt.replace(hour=rng.choice([5,6,7]))
        elif "peak" in name: dt = dt.replace(hour=rng.choice([10,11]))
        elif "post_lunch" in name: dt = dt.replace(hour=rng.choice([13,14]))

        dl_str = (dt + timedelta(hours=dl_off)).isoformat() if dl_off is not None else None
        h = dt.hour + dt.minute/60
        phase = get_phase(dt.hour)
        cog, eng = _interp(_COG, h), _interp(_NRG, h)
        urg_detail = f"deadline in {dl_off:.1f}h" if dl_off and dl_off > 0 else ("OVERDUE" if dl_off and dl_off < 0 else "none")

        tc = f"""Current time: {dt.strftime('%A, %B %d %Y at %I:%M %p')}
Circadian phase: {phase}
Cognitive capacity: {cog:.0%}
Energy level: {eng:.0%}
Urgency: {urg_detail}
Events today: {events}
Sleep last night: {sleep:.1f} hours
Weekend: {'yes' if dt.weekday()>=5 else 'no'}"""

        question, q_sub = rng.choice(QUESTIONS)
        response = generate_response(question, q_sub, dt, dl_str, events, sleep)
        system = SYSTEM_TEMPLATE.format(temporal_context=tc)

        examples.append({
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": question},
                {"role": "assistant", "content": response},
            ]
        })
    return examples


In [ ]:
# Step 3: Generate training data (3000 train + 300 eval)
train_data = generate_dataset(3000, seed=42)
eval_data = generate_dataset(300, seed=99)
print(f'Train: {len(train_data)}, Eval: {len(eval_data)}')
print(f'\nSample question: {train_data[0]["messages"][1]["content"]}')
print(f'Sample response: {train_data[0]["messages"][2]["content"][:200]}...')

In [ ]:
# Step 4: Load Gemma 3 4B with Unsloth (4-bit quantized)
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
)

# Apply LoRA
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Print trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Step 5: Prepare dataset
from datasets import Dataset

def format_chat(ex):
    text = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = Dataset.from_list(train_data).map(format_chat, remove_columns=["messages"])
eval_ds = Dataset.from_list(eval_data).map(format_chat, remove_columns=["messages"])
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")
print(f"Sample (first 300 chars): {train_ds[0]['text'][:300]}...")

In [ ]:
# Step 6: Train!
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = '/content/pulse-gemma3'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    max_seq_length=1024,
    dataset_text_field="text",
    report_to="none",
    seed=42,
    optim="adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print(f"Starting training: 3 epochs, {len(train_ds)} examples")
print(f"Steps per epoch: {len(train_ds) // (4 * 4)}")
trainer.train()

# Print final metrics
metrics = trainer.state.log_history
train_losses = [m["loss"] for m in metrics if "loss" in m]
eval_losses = [m["eval_loss"] for m in metrics if "eval_loss" in m]
if train_losses: print(f"\nFinal train loss: {train_losses[-1]:.4f}")
if eval_losses: print(f"Final eval loss: {eval_losses[-1]:.4f}")

In [ ]:
# Step 7: Save + Upload to HF Hub
from huggingface_hub import HfApi
import json as json_mod

HF_REPO = 'lalopenguin/pulse-gemma3-4b'

# Save locally
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save PULSE metadata
meta = {
    "base_model": "google/gemma-3-4b-it",
    "quantized_from": "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "pulse_version": "0.2.0",
    "training_type": "temporal_awareness_sft",
    "method": "QLoRA via Unsloth",
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 3,
    "train_examples": len(train_data),
    "eval_examples": len(eval_data),
    "max_seq_length": 1024,
}
with open(f"{OUTPUT_DIR}/pulse_config.json", "w") as f:
    json_mod.dump(meta, f, indent=2)

print(f"Model saved to {OUTPUT_DIR}")
print(f"Uploading to {HF_REPO}...")

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO,
    commit_message="PULSE temporal awareness LoRA adapter (Gemma 3 4B, Unsloth QLoRA)",
)
print(f"\nDone! https://huggingface.co/{HF_REPO}")

In [ ]:
# Step 8: Quick inference test
print("=== Test 1: Late night, sleep-deprived, imminent deadline ===")
messages = [
    {"role": "system", "content": "You have temporal awareness via PULSE. Current: Monday 2:00 AM. Circadian phase: deep_night. Cognitive capacity: 15%. Energy: 10%. Sleep last night: 4 hours. Deadline: project report in 30 minutes. Urgency: CRITICAL."},
    {"role": "user", "content": "Should I start a complex refactoring task right now?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

print("\n=== Test 2: Morning peak, well-rested, no pressure ===")
messages = [
    {"role": "system", "content": "You have temporal awareness via PULSE. Current: Tuesday 10:30 AM. Circadian phase: morning_peak. Cognitive capacity: 93%. Energy: 88%. Sleep last night: 8 hours. No deadlines. Events today: 2."},
    {"role": "user", "content": "What kind of tasks should I tackle right now?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

print("\n=== Test 3: Post-lunch dip, deadline approaching ===")
messages = [
    {"role": "system", "content": "You have temporal awareness via PULSE. Current: Wednesday 1:30 PM. Circadian phase: post_lunch_dip. Cognitive capacity: 72%. Energy: 70%. Sleep last night: 7 hours. Deadline: feature launch in 4 hours. Urgency: high."},
    {"role": "user", "content": "Should I push through or take a break?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

---

## What's next?

**To use this model locally:**
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it", torch_dtype="auto")
model = PeftModel.from_pretrained(base, "lalopenguin/pulse-gemma3-4b")
tokenizer = AutoTokenizer.from_pretrained("lalopenguin/pulse-gemma3-4b")
```

**Or with the PULSE middleware:**
```bash
pip install pulse-temporal
```
```python
from pulse_temporal import PulseMiddleware
pulse = PulseMiddleware()
system_prompt = pulse.get_temporal_system_prompt()
```

**Links:**
- [PULSE GitHub](https://github.com/lalomorales22/pulse-temporal)
- [PULSE encoder model card](https://huggingface.co/lalopenguin/pulse-base-v1)
- [Interactive demo](https://huggingface.co/spaces/lalopenguin/pulse-temporal-demo)
- [Qwen 1.5B notebook](https://colab.research.google.com/github/lalomorales22/pulse-temporal/blob/master/notebooks/train_on_colab.ipynb)